In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/raw_data/Sample - Superstore.csv", encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [10]:
# First, convert dates to actual datetime (temporarily, just for this check)
temp_order_date = pd.to_datetime(df['Order Date'])
temp_ship_date = pd.to_datetime(df['Ship Date'])

# Check for invalid logic: ship date before order date
invalid_dates = df[temp_ship_date < temp_order_date]
print("Rows where Ship Date is before Order Date:", len(invalid_dates))

Rows where Ship Date is before Order Date: 0


In [11]:
# Sales should never be zero or negative
print("Rows with Sales <= 0:", (df['Sales'] <= 0).sum())

# Quantity should never be zero or negative
print("Rows with Quantity <= 0:", (df['Quantity'] <= 0).sum())

# Discount should be between 0 and 1
print("Rows with Discount < 0 or > 1:", ((df['Discount'] < 0) | (df['Discount'] > 1)).sum())

# Profit CAN be negative - just checking the distribution, not flagging as error
print("Rows with negative Profit (expected, real losses):", (df['Profit'] < 0).sum())

Rows with Sales <= 0: 0
Rows with Quantity <= 0: 0
Rows with Discount < 0 or > 1: 0
Rows with negative Profit (expected, real losses): 1871


In [12]:
# Check for leading/trailing whitespace in key text columns
text_cols = ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Region', 
             'Category', 'Sub-Category', 'Customer Name', 'Product Name']

for col in text_cols:
    has_whitespace = df[col].astype(str).apply(lambda x: x != x.strip()).sum()
    print(f"{col}: {has_whitespace} values with leading/trailing spaces")

Ship Mode: 0 values with leading/trailing spaces
Segment: 0 values with leading/trailing spaces
Country: 0 values with leading/trailing spaces
City: 0 values with leading/trailing spaces
State: 0 values with leading/trailing spaces
Region: 0 values with leading/trailing spaces
Category: 0 values with leading/trailing spaces
Sub-Category: 0 values with leading/trailing spaces
Customer Name: 0 values with leading/trailing spaces
Product Name: 16 values with leading/trailing spaces


In [13]:
# Check casing consistency - are category values consistently capitalized?
for col in ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']:
    print(f"\n{col} unique values:")
    print(df[col].unique())


Ship Mode unique values:
['Second Class' 'Standard Class' 'First Class' 'Same Day']

Segment unique values:
['Consumer' 'Corporate' 'Home Office']

Region unique values:
['South' 'West' 'Central' 'East']

Category unique values:
['Furniture' 'Office Supplies' 'Technology']

Sub-Category unique values:
['Bookcases' 'Chairs' 'Labels' 'Tables' 'Storage' 'Furnishings' 'Art'
 'Phones' 'Binders' 'Appliances' 'Paper' 'Accessories' 'Envelopes'
 'Fasteners' 'Supplies' 'Machines' 'Copiers']


In [6]:
# Check: does each Customer ID map to exactly one Customer Name?
customer_check = df.groupby('Customer ID')['Customer Name'].nunique()
print("Customer IDs mapped to more than one name:", (customer_check > 1).sum())

# Check: does each Product ID map to exactly one Product Name?
product_check = df.groupby('Product ID')['Product Name'].nunique()
print("Product IDs mapped to more than one name:", (product_check > 1).sum())

Customer IDs mapped to more than one name: 0
Product IDs mapped to more than one name: 32


In [7]:
import numpy as np

# Using IQR method to detect outliers in Sales
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

sales_outliers = df[(df['Sales'] < lower_bound) | (df['Sales'] > upper_bound)]
print(f"Sales outliers (outside {lower_bound:.2f} to {upper_bound:.2f}):", len(sales_outliers))

# Same for Profit
Q1_p = df['Profit'].quantile(0.25)
Q3_p = df['Profit'].quantile(0.75)
IQR_p = Q3_p - Q1_p
lower_p = Q1_p - 1.5 * IQR_p
upper_p = Q3_p + 1.5 * IQR_p

profit_outliers = df[(df['Profit'] < lower_p) | (df['Profit'] > upper_p)]
print(f"Profit outliers (outside {lower_p:.2f} to {upper_p:.2f}):", len(profit_outliers))

Sales outliers (outside -271.71 to 498.93): 1167
Profit outliers (outside -39.72 to 70.82): 1881


In [9]:
df[df['Row ID'] == 7345]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
7344,7345,CA-2017-168389,12/11/2017,12/17/2017,Standard Class,DV-13045,Darrin Van Huff,Corporate,United States,Jacksonville,...,32216,South,FUR-TA-10004289,Furniture,Tables,BoxOffice By Design Rectangular and Half-Moon ...,721.875,6,0.45,-420.0
